# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their corresponding `@id` fields.

In [ ]:
print("Available Record Sets (by @id):")
record_sets = metadata.record_sets
for rs in record_sets.values():
    print(f"- {rs['@id']} : {rs['name'] if 'name' in rs else ''}")

print("\nFor each record set, fields and columns:")
for rs in record_sets.values():
    print(f"\nRecord Set @id: {rs['@id']}")
    fields = rs.get('fields') or []
    columns = rs.get('columns') or []
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - {f['@id']} : {f.get('name','')}")
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    - {c['@id']} : {c.get('name','')}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis, referencing the record set and field `@id`s found above.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in metadata.record_sets.values()]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded records for record set: {rs_id}")
        else:
            print(f"No records found for record set: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for record set: {rs_id}. Error: {e}")

# Show columns from the primary record set, for demonstration use the first record set if available
if len(dataframes) > 0:
    primary_record_set = list(dataframes.keys())[0]
    print(f"\nColumns in primary record set ({primary_record_set}):")
    print(dataframes[primary_record_set].columns.tolist())
    display(dataframes[primary_record_set].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. You should set the `numeric_field` and `group_field` to the corresponding `@id`s from the previous overview.

In [ ]:
# Example: Pick a numeric field and a group field from the primary record set
df = dataframes.get(primary_record_set)
if df is not None:
    print(f"Data types in primary record set ({primary_record_set}):")
    print(df.dtypes)
    numeric_candidates = df.select_dtypes(['number']).columns
    print("Potential numeric fields:", numeric_candidates.tolist())
    group_candidates = df.select_dtypes(['object', 'category']).columns
    print("Potential group fields:", group_candidates.tolist())

    # For demonstration, pick the first numeric column found
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 1.0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping
        if len(group_candidates) > 0:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (mean of numeric fields):")
            print(grouped_df.head())
        else:
            group_field = None
    else:
        print("No numeric fields found for analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if df is not None and len(numeric_candidates) > 0:
    field = numeric_field
    plt.figure(figsize=(8,4))
    df[field].hist(bins=30)
    plt.xlabel(field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {field}')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        df.boxplot(column=field, by=group_field)
        plt.title(f'{field} by {group_field}')
        plt.suptitle('')
        plt.ylabel(field)
        plt.xlabel(group_field)
        plt.show()
else:
    print("Insufficient data for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The `mlcroissant` library makes it easy to load and explore FAIR data described in Croissant schemas, such as this mass spectrometry dataset.
* Record sets and field structure are natively available via schema introspection (`@id` reference model).
* We demonstrated simple filtering, normalization, grouping, and visualization using automatically detected fields. Replace field names with those most relevant to your analysis as needed.
* For further analysis, consult the data dictionary and Croissant schema to select specific scientific fields and features relevant to your research question.